# Teaching Notebook: Difference-in-Differences with Fixed Effects 

This notebook demonstrates how to simulate panel data containing:

- Individual heterogeneity (unit fixed effects)
- A time shock common to all units (time fixed effects)
- A treatment applied only to treated units after a given period

We then estimate:
1. A simple OLS model  
2. A model with individual fixed effects  
3. A two-way fixed effects model (TWFE), equivalent to modern DiD regression.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_alunos = 10

ids_tratado = [f"T{i+1}" for i in range(n_alunos)]
ids_controle = [f"C{i+1}" for i in range(n_alunos)]

def gerar_linhas(id_list, tratado, efeito_tratamento=0):
    dados = []
    for aluno in id_list:
        # Individual fixed effect (alpha_i)
        efeito_fixo_individual = np.random.normal(10, 5)

        # Time shock (lambda_t)
        choque_tempo = {0: 0, 1: 7}

        # Pre period
        y_pre = 60 + efeito_fixo_individual + np.random.normal(0, 10) + choque_tempo[0]

        # Post period
        y_pos = y_pre + np.random.normal(5, 5) + choque_tempo[1]

        if tratado:
            y_pos += efeito_tratamento

        dados.append([aluno, tratado, 0, y_pre])
        dados.append([aluno, tratado, 1, y_pos])
    return dados

dados_tratado = gerar_linhas(ids_tratado, tratado=1, efeito_tratamento=10)
dados_controle = gerar_linhas(ids_controle, tratado=0)

df = pd.DataFrame(dados_tratado + dados_controle, columns=["ID", "Tratado", "Depois", "y"])
df.head()


## Exploratory Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for aluno in df.ID.unique():
    sub = df[df.ID == aluno]
    plt.plot(sub.Depois, sub.y, marker='o', alpha=0.7)

plt.xticks([0,1], ["Pré","Pós"])
plt.xlabel("Período")
plt.ylabel("Desempenho")
plt.title("Evolução individual dos alunos")
plt.show()


# Model Estimation

We estimate three models:

1. OLS without fixed effects  
2. OLS with unit fixed effects  
3. OLS with unit and time fixed effects (Two-Way Fixed Effects – TWFE)  


In [ ]:
import statsmodels.formula.api as smf

m1 = smf.ols("y ~ Tratado*Depois", data=df).fit()
print(m1.summary())


In [ ]:
m2 = smf.ols("y ~ Tratado*Depois + C(ID)", data=df).fit()
print(m2.summary())


In [ ]:
m3 = smf.ols("y ~ Tratado*Depois + C(ID) + C(Depois)", data=df).fit()
print(m3.summary())


## Group Means and DiD Visualization

In [ ]:
df_group = df.groupby(['Tratado','Depois'])['y'].mean().reset_index()

plt.figure(figsize=(8,5))
for g in [0,1]:
    sub = df_group[df_group.Tratado == g]
    plt.plot(sub.Depois, sub.y, marker='o', label=f'Tratado={g}')

plt.xticks([0,1], ['Pré','Pós'])
plt.xlabel('Período')
plt.ylabel('Média do desempenho')
plt.title('Médias pré e pós (Tratado vs Controle)')
plt.legend()
plt.show()


## Extracting Fixed Effects Estimates

In [ ]:
efeitos_fixos_unidade = m3.params.filter(like="C(ID)")
efeitos_fixos_tempo = m3.params.filter(like="C(Depois)")

efeitos_fixos_unidade, efeitos_fixos_tempo
